In [2]:
import numpy as np, pandas as pd, warnings
warnings.filterwarnings("ignore")
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score
from scipy import stats
import statsmodels.api as sm
try:
    from google.colab import files; files.upload()          # A·B·C 세 파일 업로드
except Exception: pass

a=pd.read_csv("model_A_financial_final.csv"); b=pd.read_csv("model_B_nonfinancial_final.csv"); c=pd.read_csv("model_C_nonfinancial_new_final.csv")
def clean(df):
    df=df.drop(columns=[x for x in df.columns if not pd.api.types.is_numeric_dtype(df[x])])  # AGE_BAND 등 문자열 제외
    for x in df.columns:
        if df[x].dtype==bool: df[x]=df[x].astype(int)
    return df
a,b,c=clean(a),clean(b),clean(c)
m=a.merge(b.drop(columns=["TARGET"]),on="SK_ID_CURR").merge(c.drop(columns=["TARGET"]),on="SK_ID_CURR")
y=m["TARGET"].astype(int).values
A=[x for x in a.columns if x not in("SK_ID_CURR","TARGET")]
B=[x for x in b.columns if x not in("SK_ID_CURR","TARGET")]
social=[x for x in m.columns if "SOCIAL_CIRCLE" in x]            # 사회연결망만 격리
nohist=(m["BUREAU_NO_HISTORY_FLAG"].values==1)                  # 금융이력 0/1 : 1=이력없음
AGE=m["AGE"].values
print("사회연결망 변수:", social)
print("금융이력 없음 비율:", round(nohist.mean(),3), "| 연령 범위:", round(AGE.min(),1),"~",round(AGE.max(),1))

Saving model_A_financial_final.csv to model_A_financial_final.csv
Saving model_B_nonfinancial_final.csv to model_B_nonfinancial_final.csv
Saving model_C_nonfinancial_new_final.csv to model_C_nonfinancial_new_final.csv
사회연결망 변수: ['OBS_30_CNT_SOCIAL_CIRCLE', 'DEF_30_CNT_SOCIAL_CIRCLE', 'SOCIAL_CIRCLE_MISSING_FLAG']
금융이력 없음 비율: 0.143 | 연령 범위: 20.5 ~ 69.1


In [3]:
def oof(cols):
    X=m[cols].values.astype(float); skf=StratifiedKFold(5,shuffle=True,random_state=42); o=np.zeros(len(m))
    for tr,te in skf.split(X,y):                                # 스케일링은 폴드 내부에서만 → 누수 차단
        p=Pipeline([("sc",StandardScaler()),("lr",LogisticRegression(C=1.0,max_iter=2000,class_weight="balanced",solver="lbfgs"))])
        p.fit(X[tr],y[tr]); o[te]=p.predict_proba(X[te])[:,1]
    return o
oof_base=oof(A+B); oof_soc=oof(A+B+social)
auc_b,auc_s=roc_auc_score(y,oof_base),roc_auc_score(y,oof_soc)
print(f"A+B={auc_b:.4f}  →  +사회연결망={auc_s:.4f}   전체 ΔAUC={100*(auc_s-auc_b):+.3f}%p")

A+B=0.6797  →  +사회연결망=0.6816   전체 ΔAUC=+0.192%p


In [4]:
def _mr(x):
    J=np.argsort(x);Z=x[J];N=len(x);T=np.zeros(N);i=0
    while i<N:
        j=i
        while j<N and Z[j]==Z[i]:j+=1
        T[i:j]=0.5*(i+j-1)+1.0;i=j
    o=np.empty(N);o[J]=T;return o
def _fd(p,mp):
    nt=p.shape[1];n=nt-mp;k=p.shape[0];pos,neg=p[:,:mp],p[:,mp:]
    tx=np.empty([k,mp]);ty=np.empty([k,n]);tz=np.empty([k,nt])
    for r in range(k):tx[r]=_mr(pos[r]);ty[r]=_mr(neg[r]);tz[r]=_mr(p[r])
    aucs=tz[:,:mp].sum(1)/mp/n-(mp+1.0)/2.0/n
    cov=np.cov((tz[:,:mp]-tx)/n)/mp+np.cov(1.0-(tz[:,mp:]-ty)/mp)/n
    return aucs,np.atleast_2d(cov)
def dS(mask):
    if mask.sum()==0: return None
    yy=y[mask].astype(float);o=(-yy).argsort(kind="mergesort");mp=int(yy.sum())
    if mp<25 or (len(yy)-mp)<25: return (np.nan,np.nan,int(mask.sum()))
    p=np.vstack((oof_soc[mask][o],oof_base[mask][o]));aucs,cov=_fd(p,mp)
    var=(np.array([[1.,-1.]])@cov@np.array([[1.],[-1.]])).item();d=aucs[0]-aucs[1]
    pv=1. if var<=0 else 2*(1-stats.norm.cdf(abs(d/np.sqrt(var))));return (d,pv,int(mask.sum()))

BANDS=[("10대",10,20),("20대",20,30),("30대",30,40),("40대",40,50),("50대",50,60),("60대",60,70)]
fmt=lambda r:("해당없음(n=0)" if r is None else (f"n={r[2]:,}(소표본)" if np.isnan(r[0]) else f"{r[0]*100:+.2f}%p(n={r[2]:,})"))
grid=[]
for nm,lo,hi in BANDS:
    bd=(AGE>=lo)&(AGE<hi)
    grid.append({"연령대":nm,"이력없음":fmt(dS(bd&nohist) if bd.sum() else None),
                              "이력있음":fmt(dS(bd&~nohist) if bd.sum() else None)})
grid_df=pd.DataFrame(grid); print("=== 연령대 격자 (사회연결망 ΔAUC) ===\n"+grid_df.to_string(index=False))

marg=[]
for lab,mk in [("금융이력 없음",nohist),("금융이력 있음",~nohist),("20대",(AGE>=20)&(AGE<30)),("60대",(AGE>=60)&(AGE<70))]:
    r=dS(mk); marg.append({"집단":lab,"ΔAUC(%p)":round(r[0]*100,2),"p":f"{r[1]:.1e}","n":r[2]})
marg_df=pd.DataFrame(marg); print("\n=== 주변효과 ===\n"+marg_df.to_string(index=False))

=== 연령대 격자 (사회연결망 ΔAUC) ===
연령대              이력없음              이력있음
10대         해당없음(n=0)         해당없음(n=0)
20대  +0.64%p(n=9,047) +0.14%p(n=36,090)
30대 +0.17%p(n=10,746) +0.15%p(n=71,621)
40대  +0.92%p(n=9,729) +0.37%p(n=66,852)
50대  +0.42%p(n=9,278) +0.22%p(n=58,822)
60대  +0.93%p(n=5,220) +0.09%p(n=30,106)

=== 주변효과 ===
     집단  ΔAUC(%p)       p      n
금융이력 없음      0.37 1.0e-05  44020
금융이력 있음      0.17 2.9e-08 263491
    20대      0.22 3.7e-03  45137
    60대      0.20 2.0e-01  35326


In [5]:
z=lambda s:(s-s.mean())/s.std()
X=pd.DataFrame({"DEF30":z(m["DEF_30_CNT_SOCIAL_CIRCLE"]),
                "nohist":nohist.astype(int),"young":((AGE>=20)&(AGE<30)).astype(int)})
X["DEF30_x_nohist"]=X["DEF30"]*X["nohist"]      # 가설① : 유의+양수면 지지
X["DEF30_x_young"]=X["DEF30"]*X["young"]         # 가설② : 유의+양수면 지지
res=sm.Logit(y,sm.add_constant(X)).fit(disp=0)
inter=[]
for v,lab in [("DEF30","사회연결망 부도수(주효과)"),("DEF30_x_nohist","×금융이력없음 (가설①)"),("DEF30_x_young","×20대 (가설②)")]:
    inter.append({"항":lab,"OR":round(np.exp(res.params[v]),3),"p":f"{res.pvalues[v]:.1e}",
                  "판정":"유의(지지)" if res.pvalues[v]<0.05 else "무의미(불성립)"})
inter_df=pd.DataFrame(inter); print(inter_df.to_string(index=False))

             항    OR       p       판정
사회연결망 부도수(주효과) 1.103 9.0e-46   유의(지지)
 ×금융이력없음 (가설①) 1.032 3.4e-02   유의(지지)
    ×20대 (가설②) 0.975 7.3e-02 무의미(불성립)


In [6]:
d_nh=dS(nohist)[0]; d_h=dS(~nohist)[0]
v1="성립 ✓" if (res.pvalues["DEF30_x_nohist"]<0.05 and res.params["DEF30_x_nohist"]>0) else "불성립 ✗"
v2="성립 ✓" if (res.pvalues["DEF30_x_young"]<0.05 and res.params["DEF30_x_young"]>0) else "불성립 ✗"
summ=pd.DataFrame({"항목":["가설","기법","연령대 구분","전체 사회연결망 ΔAUC","가설① 금융이력없음","가설② 나이적음(20대)","종합"],
    "결과":["금융이력 없고 나이 적을수록 사회연결망 데이터 영향 클 것","로지스틱 회귀 · 5-fold OOF · DeLong · 상호작용검정",
    "10년 단위(10대=해당없음, 20~60대)", f"+{100*(auc_s-auc_b):.3f}%p",
    f"{v1} (이력없음 {d_nh*100:+.2f}%p vs 이력있음 {d_h*100:+.2f}%p, 약2배)",
    f"{v2} (연령 무관, 상호작용 무의미)",
    "가설 절반 성립 — 금융이력 축 확인, 연령 축 기각(청년 동력은 사회연결망 아닌 학력)"]})
with pd.ExcelWriter("사회연결망_가설검정_공유.xlsx",engine="openpyxl") as xw:
    summ.to_excel(xw,sheet_name="01_요약",index=False)
    grid_df.to_excel(xw,sheet_name="02_연령대격자",index=False)
    marg_df.to_excel(xw,sheet_name="03_주변효과",index=False)
    inter_df.to_excel(xw,sheet_name="04_상호작용검정",index=False)
print("\n━━━ 팀 공유 요약 ━━━")
print(summ.to_string(index=False))
try:
    from google.colab import files; files.download("사회연결망_가설검정_공유.xlsx")
except Exception: print("saved: 사회연결망_가설검정_공유.xlsx")


━━━ 팀 공유 요약 ━━━
           항목                                                결과
           가설                  금융이력 없고 나이 적을수록 사회연결망 데이터 영향 클 것
           기법            로지스틱 회귀 · 5-fold OOF · DeLong · 상호작용검정
       연령대 구분                          10년 단위(10대=해당없음, 20~60대)
전체 사회연결망 ΔAUC                                          +0.192%p
   가설① 금융이력없음          성립 ✓ (이력없음 +0.37%p vs 이력있음 +0.17%p, 약2배)
가설② 나이적음(20대)                           불성립 ✗ (연령 무관, 상호작용 무의미)
           종합 가설 절반 성립 — 금융이력 축 확인, 연령 축 기각(청년 동력은 사회연결망 아닌 학력)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>